# Modelado — ablación de vecinos y jerarquía

Este notebook llama a `src.modeling.ablacion_espacial`. Compara la base de seis variables con los bloques de vecinos, jerarquía y ambos sobre las mismas filas. Mantiene F1 como criterio provisional para el umbral de clasificación y para seleccionar variables con las temporadas 2021–2023. Los bloques de prueba 2024 y 2025 se evalúan después de esa selección.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.ablacion_espacial import ejecutar_ablacion_espacial

reporte = ejecutar_ablacion_espacial()

h=2: 30160 filas comunes; 325 de arranque excluidas
  base_6: F1 validación reg=0.677, cls=0.592
  mas_vecinos: F1 validación reg=0.680, cls=0.593
  mas_jerarquia: F1 validación reg=0.688, cls=0.604
  mas_ambos: F1 validación reg=0.667, cls=0.595
  Elegidas h=2: {'regresion': 'mas_jerarquia', 'clasificacion': 'mas_jerarquia'}
h=4: 30030 filas comunes; 455 de arranque excluidas
  base_6: F1 validación reg=0.567, cls=0.555
  mas_vecinos: F1 validación reg=0.576, cls=0.554
  mas_jerarquia: F1 validación reg=0.586, cls=0.537
  mas_ambos: F1 validación reg=0.583, cls=0.543
  Elegidas h=4: {'regresion': 'mas_jerarquia', 'clasificacion': 'base_6'}
MLflow: run 1c12427761f14a45b79bcfab416a65a6 en 'denguekay/ablacion_espacial' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


In [2]:
import pandas as pd

filas = []
for h, bloque in reporte["horizontes"].items():
    elegidas = bloque["elegidas_solo_validacion"]
    for variante, resultado in bloque["variantes"].items():
        folds = resultado["folds_validacion"]
        filas.append({"h": int(h), "variante": variante, "n_variables": len(resultado["columnas"]),
                      "F1_regresion_validacion": sum(f["regresion"]["alerta_derivada"]["f1"] for f in folds) / len(folds),
                      "F1_clasificacion_validacion": sum(f["clasificacion"]["alerta_directa"]["f1"] for f in folds) / len(folds),
                      "elegida_regresion": variante == elegidas["regresion"],
                      "elegida_clasificacion": variante == elegidas["clasificacion"]})
display(pd.DataFrame(filas).sort_values(["h", "variante"]))

,h,variante,n_variables,F1_regresion_validacion,F1_clasificacion_validacion,elegida_regresion,elegida_clasificacion
0,2,base_6,6,0.677196,0.592203,False,False
3,2,mas_ambos,12,0.666844,0.594565,False,False
2,2,mas_jerarquia,10,0.687553,0.604039,True,True
1,2,mas_vecinos,8,0.680412,0.592572,False,False
4,4,base_6,6,0.567434,0.555295,False,True
7,4,mas_ambos,12,0.582528,0.542821,False,False
6,4,mas_jerarquia,10,0.586187,0.536682,True,False
5,4,mas_vecinos,8,0.575894,0.554143,False,False


In [3]:
filas = []
for h, bloque in reporte["horizontes"].items():
    for variante, resultado in bloque["variantes"].items():
        for fold in resultado["folds_prueba"]:
            for enfoque, metrica in (("regresion", fold["regresion"]["alerta_derivada"]),
                                     ("clasificacion", fold["clasificacion"]["alerta_directa"])):
                filas.append({"h": int(h), "bloque": fold["bloque"], "variante": variante,
                              "enfoque": enfoque, "positivos": metrica["positivos"],
                              "VP": metrica["vp"], "FP": metrica["fp"],
                              "F1": metrica["f1"]})
display(pd.DataFrame(filas).sort_values(["h", "bloque", "enfoque", "variante"]))

,h,bloque,variante,enfoque,positivos,VP,FP,F1
3,2,calendario_2025,base_6,clasificacion,3,2,282,0.013937
7,2,calendario_2025,mas_jerarquia,clasificacion,3,0,139,0.000000
2,2,calendario_2025,base_6,regresion,3,0,3,0.000000
6,2,calendario_2025,mas_jerarquia,regresion,3,0,3,0.000000
1,2,temporada_2024,base_6,clasificacion,857,789,823,0.639125
5,2,temporada_2024,mas_jerarquia,clasificacion,857,815,843,0.648111
0,2,temporada_2024,base_6,regresion,857,656,123,0.801956
4,2,temporada_2024,mas_jerarquia,regresion,857,676,154,0.801423
11,4,calendario_2025,base_6,clasificacion,3,1,349,0.005666
15,4,calendario_2025,mas_jerarquia,clasificacion,3,0,228,0.000000


Los bloques espaciales fueron seleccionados únicamente por F1 de validación. La prueba principal de 2024 ya se había inspeccionado en el experimento base; sus resultados aquí sirven como contraste de la señal nueva, no como un test completamente virgen. El informe se encuentra en `docs/modeling/ablacion_espacial.md`. Clima y demografía se revisarán en fases siguientes.